# 🍎 Fresh vs Spoiled Food Classification — PyTorch (targeting 85%+)

Updated from the 6 GB RTX notebook to fix the overfitting that capped your last run at ~78% test accuracy. Same dataset, same GPU budget.

**What's different:**

| Change | Why |
|---|---|
| Backbone B2 → **B0** | Smaller model, less memorization of small dataset |
| Image size 260 → **224** | Matches B0; less compute, more headroom |
| **MixUp + CutMix** every batch | Strongest single trick for small datasets |
| **RandAugment** added to transforms | Way more diverse than ColorJitter alone |
| **WeightedRandomSampler** | Properly balances minibatches (fixes minority classes) |
| Stratified train/val/test split | Reliable eval, especially for `fresh_dairy` |
| **EMA model** weights tracked | Free +0.5–1% accuracy at eval |
| Higher head dropout (0.5 / 0.4) | More regularization |
| Multi-scale + flip **TTA** at eval | 6 views per image, averaged |
| Longer training (20 + 25 epochs) | MixUp prevents overfitting → safe to train longer |

> ⚙️ Edit `INPUT_DIR` in cell 2 (data cleanup) before running.
> 📌 If you don't have `timm` installed already: `pip install timm`.


## 1. Setup, seeds, device, mixed precision

In [ ]:
import os, math, random, shutil, time, gc, copy
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder

import timm
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy
from timm.utils import ModelEmaV3

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print('PyTorch:    ', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('timm:       ', timm.__version__)

SEED = 1337
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device('cuda')
    props = torch.cuda.get_device_properties(0)
    print(f'✅ GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
else:
    device = torch.device('cpu')
    print('⚠️  No GPU — this will be very slow.')

USE_AMP = device.type == 'cuda'
USE_CHANNELS_LAST = device.type == 'cuda' and torch.cuda.get_device_capability(0)[0] >= 8

def vram_report(tag=''):
    if device.type != 'cuda': return
    alloc = torch.cuda.memory_allocated() / 1e9
    peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f'   [VRAM{(" " + tag) if tag else ""}] allocated={alloc:.2f}GB  peak={peak:.2f}GB')

def vram_reset():
    if device.type != 'cuda': return
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

vram_report('startup')


## 2. Data cleanup

In [ ]:
# 👉 EDIT THIS to your local dataset path
INPUT_DIR   = Path(r'./fresh-and-spoiled-food-image-dataset/dataset')
WORKING_DIR = Path(r'./clean_dataset')

print('1) Refreshing the working directory...')
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)
shutil.copytree(INPUT_DIR, WORKING_DIR)

print('2) PIL image scan...')
bad = 0
for p in WORKING_DIR.rglob('*'):
    if not p.is_file(): continue
    try:
        with Image.open(p) as img: img.verify()
        with Image.open(p) as img: img.convert('RGB')
    except Exception:
        p.unlink(); bad += 1
        print('  removed:', p.name)
print(f'✅ Cleanup complete. Removed {bad} incompatible files.')


## 3. Datasets — stratified 80/10/10 split + strong augmentation

In [ ]:
DATA_DIR        = WORKING_DIR
IMG_SIZE        = 224         # B0's native res; smaller img → less overfit headroom
BATCH_SIZE      = 32
EVAL_BATCH_SIZE = 64
NUM_WORKERS     = 2

IM_MEAN = [0.485, 0.456, 0.406]
IM_STD  = [0.229, 0.224, 0.225]

# Strong train-time augmentation (RandAugment is the new piece)
train_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IM_MEAN, IM_STD),
])

ds_train_view = ImageFolder(str(DATA_DIR), transform=train_tf)
ds_eval_view  = ImageFolder(str(DATA_DIR), transform=eval_tf)

class_names = ds_train_view.classes
num_classes = len(class_names)
all_labels  = np.array(ds_train_view.targets)
print(f'✅ {num_classes} classes: {class_names}')
print(f'Total images: {len(ds_train_view)}')

# Stratified 80/10/10 split — every class proportionally represented in each split
indices = np.arange(len(all_labels))
train_idx, temp_idx, train_lbl, temp_lbl = train_test_split(
    indices, all_labels, test_size=0.2, stratify=all_labels, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5, stratify=temp_lbl, random_state=SEED
)

train_ds = Subset(ds_train_view, train_idx)
val_ds   = Subset(ds_eval_view,  val_idx)
test_ds  = Subset(ds_eval_view,  test_idx)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

print('\nPer-class counts (train | val | test):')
print(f'{"class":<22}{"train":>8}{"val":>6}{"test":>6}')
for c in range(num_classes):
    tr = int((all_labels[train_idx] == c).sum())
    va = int((all_labels[val_idx]   == c).sum())
    te = int((all_labels[test_idx]  == c).sum())
    print(f'{class_names[c]:<22}{tr:>8}{va:>6}{te:>6}')


### 3a. WeightedRandomSampler — balance every minibatch

Class weights in the loss only re-weight the gradient signal. A `WeightedRandomSampler` actually changes which samples appear in each batch, so every minibatch is approximately balanced. This is a much stronger fix for `fresh_dairy` than `weight=` alone.

In [ ]:
# Per-sample weight = 1 / (class frequency in train set)
train_targets = all_labels[train_idx]
class_count   = np.bincount(train_targets, minlength=num_classes)
class_weight_arr = 1.0 / class_count

sample_weights = class_weight_arr[train_targets]
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(train_idx), replacement=True
)

print('Per-class weighting (smaller class → larger weight):')
for c in range(num_classes):
    print(f'  {class_names[c]:<22}  train_count={class_count[c]:>4}  weight={class_weight_arr[c]:.4f}')

pin = device.type == 'cuda'
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=NUM_WORKERS > 0,
)


### 3b. Quick visual sanity check

In [ ]:
def denorm(t):
    mean = torch.tensor(IM_MEAN).view(3, 1, 1)
    std  = torch.tensor(IM_STD).view(3, 1, 1)
    return (t.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

x_sample, y_sample = next(iter(train_loader))
plt.figure(figsize=(10, 10))
for i in range(min(9, len(x_sample))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(denorm(x_sample[i]))
    plt.title(class_names[int(y_sample[i])])
    plt.axis('off')
plt.tight_layout(); plt.show()


## 4. MixUp + CutMix, then EfficientNetV2-**B0** + custom head

MixUp blends two images and their labels: `image = λ·xa + (1-λ)·xb`. CutMix pastes a patch from one image onto another. Both produce **soft labels**, so we use `SoftTargetCrossEntropy` instead of regular cross-entropy.

The backbone is B0 (7M params) instead of B2 — less capacity to memorize the small dataset. Head dropout is bumped to **0.5 / 0.4** for the same reason.

In [ ]:
# ----- MixUp / CutMix -----
mixup_fn = Mixup(
    mixup_alpha   = 0.8,
    cutmix_alpha  = 1.0,
    cutmix_minmax = None,
    prob          = 1.0,     # always apply one
    switch_prob   = 0.5,     # 50/50 between mixup and cutmix
    mode          = 'batch',
    label_smoothing = 0.1,
    num_classes   = num_classes,
)
print('✅ MixUp + CutMix enabled')

# ----- Model -----
BACKBONE = 'tf_efficientnetv2_b0'

class FoodClassifier(nn.Module):
    def __init__(self, num_classes, backbone_name=BACKBONE, pretrained=True,
                 dropout_head=0.5, dropout_neck=0.4):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            num_classes=0, global_pool='avg',
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout_head),
            nn.Linear(feat_dim, 256),
            nn.SiLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_neck),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_top(self, freeze_ratio=0.5):
        for p in self.backbone.parameters(): p.requires_grad = True
        blocks = self.backbone.blocks
        freeze_until = int(len(blocks) * freeze_ratio)
        for i, blk in enumerate(blocks):
            if i < freeze_until:
                for p in blk.parameters(): p.requires_grad = False
        for name, p in self.backbone.named_parameters():
            if name.startswith('conv_stem') or name.startswith('bn1'):
                p.requires_grad = False
        for m in self.backbone.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                for p in m.parameters(): p.requires_grad = False


def set_bn_eval(module):
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
            m.eval()


vram_reset()
model = FoodClassifier(num_classes=num_classes).to(device)
if USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)
    print('Using channels_last memory format')

print(f'Total params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

# EMA — exponential moving average of weights. Almost always evaluates better.
ema_model = ModelEmaV3(model, decay=0.9998, device=device)
print('✅ EMA model created (decay=0.9998)')
vram_report('after model + EMA on GPU')


## 5. Train / evaluate loops (MixUp-aware + EMA)

In [ ]:
def make_amp():
    if not USE_AMP:
        return (lambda: torch.cuda.amp.autocast(enabled=False)), None
    try:
        from torch.amp import autocast as _ac, GradScaler as _GS
        return (lambda: _ac(device_type='cuda', dtype=torch.float16)), _GS('cuda')
    except (ImportError, TypeError):
        from torch.cuda.amp import autocast as _ac, GradScaler as _GS
        return (lambda: _ac()), _GS()


def _to_device(x):
    x = x.to(device, non_blocking=True)
    if USE_CHANNELS_LAST and x.dim() == 4:
        x = x.contiguous(memory_format=torch.channels_last)
    return x


def train_one_epoch(model, loader, criterion, optimizer, amp_ctx, scaler,
                    mixup_fn=None, ema=None, bn_frozen=False, desc='train'):
    model.train()
    if bn_frozen:
        set_bn_eval(model.backbone)

    total_loss, total_n = 0.0, 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for x, y in pbar:
        x = _to_device(x); y = y.to(device, non_blocking=True)

        if mixup_fn is not None:
            x, y = mixup_fn(x, y)   # x mixed, y soft target (B, C)

        optimizer.zero_grad(set_to_none=True)
        with amp_ctx():
            logits = model(x)
            loss   = criterion(logits, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()

        if ema is not None:
            ema.update(model)

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_n    += bs
        pbar.set_postfix(loss=f'{total_loss/total_n:.4f}')
    return total_loss / max(1, total_n)


@torch.no_grad()
def evaluate(model, loader, amp_ctx, desc='eval'):
    """Eval on integer labels (no mixup at eval time)."""
    model.eval()
    correct, total = 0, 0
    for x, y in tqdm(loader, desc=desc, leave=False):
        x = _to_device(x); y = y.to(device, non_blocking=True)
        with amp_ctx():
            logits = model(x)
        correct += (logits.argmax(1) == y).sum().item()
        total   += x.size(0)
    return correct / max(1, total)


class EarlyStopping:
    def __init__(self, patience=8, mode='max'):
        self.patience = patience; self.mode = mode
        self.best = -math.inf if mode == 'max' else math.inf
        self.bad  = 0
    def step(self, value):
        improved = value > self.best if self.mode == 'max' else value < self.best
        if improved: self.best = value; self.bad = 0
        else: self.bad += 1
        return self.bad >= self.patience


def run_phase(model, ema_model, train_loader, val_loader, *,
              epochs, lr_max, weight_decay, warmup_epochs,
              ckpt_path, mixup_fn, bn_frozen=False, patience=10):
    amp_ctx, scaler = make_amp()

    criterion = SoftTargetCrossEntropy() if mixup_fn is not None \
                else LabelSmoothingCrossEntropy(smoothing=0.1)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(params, lr=lr_max, weight_decay=weight_decay)

    warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                      total_iters=max(1, warmup_epochs))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, epochs - warmup_epochs),
                               eta_min=lr_max * 0.01)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[max(1, warmup_epochs)])

    es = EarlyStopping(patience=patience, mode='max')
    history = {'train_loss': [], 'val_acc': [], 'ema_val_acc': [], 'lr': []}
    best_val_acc = -1.0

    for epoch in range(epochs):
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'\nEpoch {epoch+1}/{epochs}  lr={cur_lr:.2e}')

        tr_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, amp_ctx, scaler,
            mixup_fn=mixup_fn, ema=ema_model, bn_frozen=bn_frozen,
            desc=f'ep {epoch+1} train',
        )
        va_acc     = evaluate(model,            val_loader, amp_ctx, desc=f'ep {epoch+1} val')
        ema_va_acc = evaluate(ema_model.module, val_loader, amp_ctx, desc=f'ep {epoch+1} ema val')

        history['train_loss'].append(tr_loss)
        history['val_acc'].append(va_acc)
        history['ema_val_acc'].append(ema_va_acc)
        history['lr'].append(cur_lr)

        print(f'  train loss={tr_loss:.4f}')
        print(f'  val acc (raw) ={va_acc:.4f}   |   val acc (EMA) ={ema_va_acc:.4f}')

        best_this_epoch = max(va_acc, ema_va_acc)
        save_ema = ema_va_acc >= va_acc
        if best_this_epoch > best_val_acc:
            best_val_acc = best_this_epoch
            state = (ema_model.module if save_ema else model).state_dict()
            torch.save({
                'model_state_dict': state,
                'class_names': class_names,
                'epoch': epoch,
                'val_acc': best_this_epoch,
                'used_ema': save_ema,
            }, ckpt_path)
            tag = 'EMA' if save_ema else 'raw'
            print(f'  ✅ new best val_acc ({tag}) → saved to {ckpt_path}')

        scheduler.step()

        if es.step(best_this_epoch):
            print(f'  ⏹ Early stopping (patience={patience}).')
            break

    return history, best_val_acc


## 6. Phase 1 — train the head (backbone frozen)

Lower LR (`3e-4`) and more epochs (20). With MixUp providing soft labels, an aggressive LR makes the loss noisy.

In [ ]:
EPOCHS_PHASE1 = 20
LR_PHASE1     = 3e-4

model.freeze_backbone()
print(f'Phase 1 trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.3f}M')

vram_reset()
t0 = time.time()
history1, best1 = run_phase(
    model, ema_model, train_loader, val_loader,
    epochs=EPOCHS_PHASE1, lr_max=LR_PHASE1, weight_decay=1e-4,
    warmup_epochs=2, ckpt_path='best_phase1.pt',
    mixup_fn=mixup_fn, bn_frozen=False, patience=8,
)
print(f'\nPhase 1 done in {(time.time()-t0)/60:.1f} min — best val_acc={best1:.4f}')
vram_report('end of Phase 1')


## 7. Phase 2 — fine-tune top half (BN frozen, EMA reset)

The EMA is reset before Phase 2 so it tracks from the new starting point. Otherwise it would carry stale weights from Phase 1's frozen-backbone regime.

In [ ]:
EPOCHS_PHASE2 = 25
LR_PHASE2     = 5e-5

ckpt = torch.load('best_phase1.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
ema_model = ModelEmaV3(model, decay=0.9998, device=device)
print(f'Loaded Phase 1 best (val_acc={ckpt["val_acc"]:.4f}, used_ema={ckpt["used_ema"]})')

model.unfreeze_top(freeze_ratio=0.5)
print(f'Phase 2 trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.3f}M')

vram_reset()
t0 = time.time()
history2, best2 = run_phase(
    model, ema_model, train_loader, val_loader,
    epochs=EPOCHS_PHASE2, lr_max=LR_PHASE2, weight_decay=1e-5,
    warmup_epochs=2, ckpt_path='best_finetuned.pt',
    mixup_fn=mixup_fn, bn_frozen=True, patience=10,
)
print(f'\nPhase 2 done in {(time.time()-t0)/60:.1f} min — best val_acc={best2:.4f}')
vram_report('end of Phase 2')

ckpt = torch.load('best_finetuned.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'\n✅ Loaded best fine-tuned weights (val_acc={ckpt["val_acc"]:.4f}, used_ema={ckpt["used_ema"]})')


## 8. Training curves

In [ ]:
val_acc     = history1['val_acc']     + history2['val_acc']
ema_val_acc = history1['ema_val_acc'] + history2['ema_val_acc']
losses      = history1['train_loss']  + history2['train_loss']
boundary    = len(history1['val_acc']) - 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(val_acc,     label='val (raw)', marker='o', ms=3)
axes[0].plot(ema_val_acc, label='val (EMA)', marker='s', ms=3)
axes[0].axvline(boundary, color='gray', ls='--', label='start fine-tune')
axes[0].set_title('Validation accuracy')
axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(losses, label='train loss', marker='o', ms=3, color='tab:red')
axes[1].axvline(boundary, color='gray', ls='--', label='start fine-tune')
axes[1].set_title('Training loss')
axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()


## 9. Evaluate on the held-out test set + multi-scale TTA

3 crop scales × {original, horizontal flip} = 6 views per image, softmaxes averaged.

In [ ]:
amp_ctx, _ = make_amp()

# Plain test eval
test_acc_plain = evaluate(model, test_loader, amp_ctx, desc='test (no TTA)')
print(f'\n📊 Test accuracy (no TTA):              {test_acc_plain*100:.2f}%')

# Multi-scale + flip TTA
@torch.no_grad()
def predict_multiscale_tta(model, loader, amp_ctx, scales=(1.0, 1.07, 1.14)):
    model.eval()
    all_proba, all_y = [], []
    for x, y in tqdm(loader, desc='TTA', leave=False):
        x = _to_device(x)
        probas = []
        for s in scales:
            if s == 1.0:
                x_s = x
            else:
                new_h = int(x.shape[2] * s); new_w = int(x.shape[3] * s)
                x_s = F.interpolate(x, size=(new_h, new_w), mode='bilinear', align_corners=False)
                top  = (new_h - IMG_SIZE) // 2
                left = (new_w - IMG_SIZE) // 2
                x_s  = x_s[:, :, top:top+IMG_SIZE, left:left+IMG_SIZE]
            with amp_ctx():
                probas.append(F.softmax(model(x_s),                       dim=1))
                probas.append(F.softmax(model(torch.flip(x_s, dims=[3])), dim=1))
        avg = torch.stack(probas).mean(dim=0)
        all_proba.append(avg.cpu().float().numpy())
        all_y.append(y.numpy())
    return np.concatenate(all_proba), np.concatenate(all_y)

y_proba, y_true = predict_multiscale_tta(model, test_loader, amp_ctx)
y_pred = y_proba.argmax(axis=1)
test_acc_tta = float((y_pred == y_true).mean())
print(f'📊 Test accuracy (multi-scale + flip TTA): {test_acc_tta*100:.2f}%')

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix (test set, TTA) — acc = {test_acc_tta*100:.2f}%')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

print('\nClassification Report (test set, TTA):\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


## 10. Save the final model — overwrites the file your Flask app loads

In [ ]:
FINAL_PATH = 'fresh_spoiled_food_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names':      class_names,
    'backbone_name':    BACKBONE,
    'img_size':         IMG_SIZE,
    'normalize_mean':   IM_MEAN,
    'normalize_std':    IM_STD,
    'test_acc_no_tta':  test_acc_plain,
    'test_acc_tta':     test_acc_tta,
}, FINAL_PATH)
print(f'✅ Saved: {FINAL_PATH}')
print(f'   Final test accuracy: {test_acc_tta*100:.2f}%  (TTA)')

# Sanity reload
ckpt = torch.load(FINAL_PATH, map_location=device)
reload_model = FoodClassifier(num_classes=len(ckpt['class_names']),
                              backbone_name=ckpt['backbone_name'],
                              pretrained=False).to(device)
reload_model.load_state_dict(ckpt['model_state_dict'])
reload_model.eval()
print('Reload OK. Classes:', ckpt['class_names'])


## 11. Predict on a single image

In [ ]:
@torch.no_grad()
def predict_image(path: str, tta: bool = True):
    img = Image.open(path).convert('RGB')
    x = eval_tf(img).unsqueeze(0)
    x = _to_device(x)
    model.eval()
    amp_ctx, _ = make_amp()
    with amp_ctx():
        p = F.softmax(model(x), dim=1)
        if tta:
            p = (p + F.softmax(model(torch.flip(x, dims=[3])), dim=1)) / 2
    p = p.cpu().float().numpy()[0]
    idx = int(p.argmax())
    return class_names[idx], float(p[idx]), {c: float(p[i]) for i, c in enumerate(class_names)}

# Example:
# label, conf, all_probs = predict_image('path/to/apple.jpg')
# print(f'{label}  ({conf*100:.1f}%)')


## 12. If you're still below 85%

Try these **one at a time** — don't change them all at once.

1. **Train longer**: bump `EPOCHS_PHASE2` to 35 and `patience` to 15. MixUp prevents overfitting, so longer training is safe.
2. **Switch to B1**: in cell 12, set `BACKBONE = 'tf_efficientnetv2_b1'`. More capacity but still small enough for your dataset.
3. **Stronger MixUp**: in cell 12, bump `mixup_alpha=1.0, cutmix_alpha=1.2`.
4. **Two-model ensemble**: train this notebook twice with different seeds (`SEED=1337` then `SEED=42`), save both, average their TTA softmaxes.
5. **Look at the confusion matrix**: if `fresh_dairy` is still the worst, you may need to collect more `fresh_dairy` images — the model can only do so much with 166 training images of that class.

If you OOM (unlikely with B0 at 224), drop `BATCH_SIZE` to 16 and `EVAL_BATCH_SIZE` to 32.


---

*Notebook updated to target 85%+ accuracy. Original 6 GB RTX version's techniques (mixed precision, channels_last, BN freezing during fine-tune) all retained.*

In [ ]:
# (this cell was a stray import from a previous run — left empty)
